# 카카오 API - 한국 소설 메타데이터 수집

카카오 책 검색 API를 사용해 한국 소설 메타데이터를 수집하고 JSON으로 저장합니다.

**수집 대상 필드**: 제목, 저자, 출판사, 출판일, ISBN, 소개글, 썸네일, URL

In [ ]:
# 필요 라이브러리 설치
!pip install requests pandas tqdm python-dotenv -q

In [ ]:
import requests
import pandas as pd
import json
import time
from tqdm import tqdm
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"), override=True)

## 1. 설정

In [ ]:
# =====================
# 설정값
# =====================
API_KEY = os.getenv("Kakao_api")   # 카카오 REST API 키
BASE_URL = "https://dapi.kakao.com/v3/search/book"
OUTPUT_DIR = Path("../data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 소설 관련 검색 키워드 (2500건 제한 우회용)
SEARCH_KEYWORDS = [
    "현대소설", "한국소설", "장편소설", "단편소설",
    "추리소설", "역사소설", "판타지소설", "로맨스소설",
    "SF소설", "공포소설", "청소년소설"
]

PAGE_SIZE = 50   # 카카오 API 최대 50건/페이지
SLEEP_SEC = 0.3  # API 호출 간격 (서버 부하 방지)
print("API KEY 로드:", "OK" if API_KEY else "없음 - .env 확인 필요")

## 2. API 호출 함수

In [ ]:
def fetch_novels(keyword: str, page_num: int = 1, retries: int = 3) -> dict:
    headers = {"Authorization": f"KakaoAK {API_KEY}"}
    params = {
        "query": keyword,
        "sort": "latest",
        "page": page_num,
        "size": PAGE_SIZE,
        "target": "title",
    }
    for attempt in range(retries):
        try:
            resp = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as e:
            print(f"[재시도 {attempt+1}/{retries}] 키워드: {keyword}, 페이지: {page_num} → {e}")
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
    return {}


def parse_result(data: dict) -> list[dict]:
    records = []
    for item in data.get("documents", []):
        records.append({
            "title":      item.get("title", "").strip(),
            "author":     ", ".join(item.get("authors", [])),
            "publisher":  item.get("publisher", "").strip(),
            "pub_date":   item.get("datetime", "")[:10],   # YYYY-MM-DD
            "isbn":       item.get("isbn", "").strip(),
            "contents":   item.get("contents", "").strip(),
            "thumbnail":  item.get("thumbnail", "").strip(),
            "url":        item.get("url", "").strip(),
            "translators": ", ".join(item.get("translators", [])),
            "status":     item.get("status", "").strip(),
        })
    return records

## 3. 전체 수집 실행

In [ ]:
# [선택] 응답 구조 확인용 디버깅 셀 - 실제 API를 1회 호출합니다
# 구조를 이미 확인했다면 이 셀은 건너뛰세요
data = fetch_novels("현대소설", page_num=1)
print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])

In [ ]:
all_records = []
seen_isbns = set()  # ISBN 기준 중복 제거

for keyword in SEARCH_KEYWORDS:
    print(f"\n📚 키워드: [{keyword}] 수집 시작")
    page = 1

    while True:
        data = fetch_novels(keyword, page)

        meta = data.get("meta", {})
        records = parse_result(data)

        if not records:
            print(f"  ✅ {page-1}페이지까지 수집 완료")
            break

        # 중복 제거
        new_count = 0
        for r in records:
            isbn = r["isbn"]
            if isbn and isbn not in seen_isbns:
                seen_isbns.add(isbn)
                all_records.append(r)
                new_count += 1
            elif not isbn:  # ISBN 없는 경우 제목+저자로 중복 판단
                key = r["title"] + r["author"]
                if key not in seen_isbns:
                    seen_isbns.add(key)
                    all_records.append(r)
                    new_count += 1

        total = meta.get("pageable_count", 0)
        is_end = meta.get("is_end", True)
        print(f"  페이지 {page} | 신규 {new_count}건 | 누적 {len(all_records)}건 (노출 가능 {total}건)")

        # 마지막 페이지 체크 (카카오 API: 최대 50페이지)
        if is_end or page >= 50:
            break

        page += 1
        time.sleep(SLEEP_SEC)

print(f"\n🎉 수집 완료 | 총 {len(all_records)}건 (중복 제거 후)")

## 4. 저장

In [ ]:
# JSON 저장
output_path = OUTPUT_DIR / "novels_raw.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"✅ JSON 저장 완료: {output_path}")

# DataFrame 확인
df = pd.DataFrame(all_records)
print(f"\n📊 데이터 형태: {df.shape}")
df.head(5)


## 5. 데이터 탐색

In [ ]:
# 출판일 분포
print("=== 출판년도 분포 (상위 10) ===")
print(df["pub_date"].str[:4].value_counts().head(10))

print("\n=== 출판사 분포 (상위 10) ===")
print(df["publisher"].value_counts().head(10))

print("\n=== 결측값 현황 ===")
print(df.isnull().sum())

print("\n=== 판매 상태 분포 ===")
print(df["status"].value_counts())

In [ ]:
# CSV로도 저장 (선택)
csv_path = OUTPUT_DIR / "novels_raw.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ CSV 저장 완료: {csv_path}")
